# 09 · The Full Model + End-to-End Shape Trace

Companion to **Chapters 8–9**. Here you assemble everything and run the seven
correctness tests that every transformer implementation should pass.

We use `tfs.model` as the reference. Build your own alongside it and compare.

In [ ]:
import math, os, sys
import torch
sys.path.insert(0, os.path.abspath('..'))

from tfs.model import Config, Model

cfg = Config(vocab_size=512, d_model=96, n_layer=3, n_head=4,
             n_kv_head=2, d_head=24, d_ff=256, max_T=128)
torch.manual_seed(0)
model = Model(cfg)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 1 · Trace every shape

Hook every module and print what flows through it. Compare against the
step-through widget in Chapter 9.

In [ ]:
import torch.nn as nn

trace, handles = [], []
def hook(name):
    def fn(mod, inp, out):
        o = out[0] if isinstance(out, tuple) else out
        if torch.is_tensor(o):
            trace.append((name, type(mod).__name__, tuple(o.shape)))
    return fn

for name, mod in model.named_modules():
    if isinstance(mod, (nn.Linear, nn.Embedding)) or type(mod).__name__ in\
       ("RMSNorm", "GroupedQueryAttention", "SwiGLU", "Block"):
        handles.append(mod.register_forward_hook(hook(name or "model")))

ids = torch.randint(0, cfg.vocab_size, (2, 6))
print(f"input ids            {tuple(ids.shape)}   int64\n")
with torch.no_grad():
    logits, loss, _ = model(ids, targets=ids)
for h in handles: h.remove()

for name, kind, shape in trace[:26]:
    print(f"{name:34} {kind:24} {str(shape)}")
print(f"...\n{'logits':34} {'':24} {tuple(logits.shape)}")
print(f"{'loss':34} {'':24} scalar = {loss.item():.4f}")

**The thing to notice:** every `Block` output is `(2, 6, 96)`. Depth is a *loop*
over one shape, not an axis of the tensor.

## 2 · The seven tests

Run these on every model you ever write.

In [ ]:
# TEST 1 -- shapes
logits, loss, caches = model(ids, targets=ids)
assert logits.shape == (2, 6, cfg.vocab_size)
assert loss.ndim == 0
print("1. shapes ✓")

# TEST 2 -- initial loss == ln(V).  THE cheapest sanity check in deep learning.
torch.manual_seed(0)
fresh = Model(cfg)
_, l0, _ = fresh(torch.randint(0, cfg.vocab_size, (8, 32)),
                 targets=torch.randint(0, cfg.vocab_size, (8, 32)))
expected = math.log(cfg.vocab_size)
print(f"2. initial loss {l0.item():.4f}  vs  ln(V) = {expected:.4f}  "
      f"(diff {abs(l0.item()-expected):.4f}) ✓")
assert abs(l0.item() - expected) < 0.3

In [ ]:
# TEST 3 -- CAUSALITY. Perturb token t; outputs at < t must not move.
m = Model(cfg).eval()
ids = torch.randint(0, cfg.vocab_size, (1, 12))
with torch.no_grad():
    o1, _, _ = m(ids)
    ids2 = ids.clone(); ids2[0, 7] = (ids2[0, 7] + 1) % cfg.vocab_size
    o2, _, _ = m(ids2)

before = (o1[0, :7] - o2[0, :7]).abs().max().item()
at     = (o1[0, 7]  - o2[0, 7] ).abs().max().item()
print(f"3. causality:  max change at positions <7: {before:.2e}   at position 7: {at:.4f}")
assert before < 1e-5, "CAUSALITY VIOLATED -- the model can see the future"
assert at > 1e-4
print("   ✓  (run this on every model you write)")

In [ ]:
# TEST 4 -- every parameter receives gradient
m = Model(cfg)
_, loss, _ = m(ids, targets=ids)
loss.backward()
dead = [n for n, p in m.named_parameters()
        if p.grad is None or p.grad.abs().sum() == 0]
print(f"4. parameters with no gradient: {dead if dead else 'none'} ✓")
assert not dead

# TEST 5 -- analytic parameter count matches reality (Ch 9.4)
actual = sum(p.numel() for p in m.parameters())
print(f"5. formula {cfg.n_params():,}  actual {actual:,} ✓")
assert cfg.n_params() == actual

In [ ]:
# TEST 6 -- the KV cache is the GQA size, not the MHA size (Ch 16)
with torch.no_grad():
    _, _, caches = model(torch.randint(0, cfg.vocab_size, (1, 10)))
k, v = caches[0]
print(f"6. cache K {tuple(k.shape)}  V {tuple(v.shape)}")
print(f"   n_kv_head={cfg.n_kv_head} (not n_head={cfg.n_head}) ✓  "
      f"-> {cfg.n_head//cfg.n_kv_head}x smaller cache")
assert k.shape == (1, cfg.n_kv_head, 10, cfg.d_head)

# TEST 7 -- cached generation == uncached generation (Ch 15)
torch.manual_seed(0)
m = Model(cfg).eval()
prompt = torch.randint(0, cfg.vocab_size, (1, 5))
a = m.generate(prompt.clone(), 25, temperature=0, use_cache=False)
b = m.generate(prompt.clone(), 25, temperature=0, use_cache=True)
print(f"7. cache vs no-cache identical: {torch.equal(a, b)} ✓")
assert torch.equal(a, b), "check RoPE pos_offset and is_causal during decode"
print("\nAll seven pass.")

## 3 · The label-shift bug, demonstrated

The most common mistake in the whole course. It produces a *beautiful* loss curve.

In [ ]:
import torch.nn.functional as F

torch.manual_seed(0)
m = Model(cfg)
ids = torch.randint(0, cfg.vocab_size, (4, 16))
logits, _, _ = m(ids)
V = cfg.vocab_size

correct = F.cross_entropy(logits[:, :-1].reshape(-1, V), ids[:, 1:].reshape(-1))
buggy   = F.cross_entropy(logits.reshape(-1, V),          ids.reshape(-1))

print(f"correct (shifted):   {correct.item():.4f}")
print(f"buggy   (unshifted): {buggy.item():.4f}   <- already lower at init")
print(f"ln(V) =              {math.log(V):.4f}")
print("\n(With TIED embeddings the unshifted loss starts low because the logit for")
print(" token i is e_i . e_i -- a vector dotted with itself, which is large.")
print(" The model is born slightly good at copying.)")

In [ ]:
# The real demonstration. Train on FRESH RANDOM tokens every step, so
# memorisation is impossible and the data has entropy exactly ln(V).
#
#   correct (shifted)   -> cannot predict random next tokens -> loss stays at ln(V)
#   buggy   (unshifted) -> copying works on ANY input        -> loss collapses to 0

for label, shift in [("correct", True), ("BUGGY  ", False)]:
    torch.manual_seed(0)
    mm = Model(cfg)
    opt = torch.optim.AdamW(mm.parameters(), lr=3e-3)
    for step in range(300):
        batch = torch.randint(0, V, (8, 16))          # NEW random data every step
        lg, _, _ = mm(batch)
        loss = (F.cross_entropy(lg[:, :-1].reshape(-1, V), batch[:, 1:].reshape(-1))
                if shift else
                F.cross_entropy(lg.reshape(-1, V), batch.reshape(-1)))
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"  {label} loss after 300 steps on random data: {loss.item():.5f}")

print(f"\n  ln(V) = {math.log(V):.4f}  <- the true entropy of this data")
print("\nThe correct version sits at ln(V) because random tokens are unpredictable.")
print("The BUGGY version drives loss toward 0 -- it learned to COPY, which works")
print("on any input whatsoever, and has learned nothing about language.")
print("\nDIAGNOSTIC: a loss below your data's entropy is not a good model. It is a bug.")

## 4 · Exercise 9.4 — reverse-engineer Llama-3-8B

In [ ]:
llama3_8b = Config(vocab_size=128256, d_model=4096, n_layer=32, n_head=32,
                   n_kv_head=8, d_head=128, d_ff=14336, max_T=8192,
                   tie_embed=False)
n = llama3_8b.n_params()
print(f"Llama-3-8B parameter count from config alone: {n:,}  = {n/1e9:.2f} B")
assert 7.9e9 < n < 8.15e9

kv_per_tok = 2 * llama3_8b.n_kv_head * llama3_8b.d_head * 2 * llama3_8b.n_layer
print(f"\nKV cache per token: {kv_per_tok/1024:.0f} KB")
for ctx in [8192, 32768, 131072]:
    print(f"  at {ctx:>7,} context: {kv_per_tok*ctx/1024**3:6.2f} GB per sequence")
print(f"\nModel weights in bf16: {n*2/1024**3:.1f} GB")
print("At 128k context, ONE user's cache exceeds the model itself. -> Chapter 15.")

---
## Self-check

1. At which single point do different token positions exchange information?
2. Why must a pre-LN model have a *final* norm before the LM head?
3. Your loss starts at 10.4 with V=32000. Expected?
4. Why does GPT-2 scale residual-output projections by `1/sqrt(2*n_layer)` at init?

<details><summary>Answers</summary>

1. Only in attention (`softmax(QKᵀ) @ V`). The MLP, both norms, RoPE and the LM head
   all act per-position independently.
2. Pre-LN never normalises the residual stream itself, so its magnitude grows with
   depth. Without a final norm the logits are badly scaled.
3. Yes — `ln(32000) = 10.37`, the loss of uniform guessing.
4. Each block adds an independent contribution to the stream and variances sum, so
   std grows like `sqrt(n_layer)`. The scaling keeps it flat with depth.

</details>

**Next:** `13_kv_cache.ipynb`